# 08 retention prioritization

Probability × future gross revenue describes economic exposure, not causal treatment benefit.

Reusable calculations live in src/ and run_pipeline.py. This notebook reads executed artifacts so analysis cannot silently retrain or leak future data.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
results=json.loads((root/'artifacts/tables/results.json').read_text())

In [2]:
data=pd.read_csv(root/'artifacts/tables/segments.csv')
display(data.head(12))
display(data.describe(include='all').T)

,segment,customers,mean_risk,predicted_revenue,observed_future_revenue
0,High value / High risk,26,0.595957,10546.421907,7506.29
1,High value / Low risk,156,0.231578,178184.300196,125825.89
2,Low value / High risk,156,0.696202,32125.986327,31976.75
3,Low value / Low risk,25,0.383314,6406.578557,2668.68


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
segment,4,4,High value / High risk,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customers,4.0,NaN,NaN,NaN,90.75,75.345316,25.0,25.75,91.0,156.0,156.0
mean_risk,4.0,NaN,NaN,NaN,0.476763,0.209133,0.231578,0.34538,0.489636,0.621018,0.696202
predicted_revenue,4.0,NaN,NaN,NaN,56815.821747,81694.23622,6406.578557,9511.461069,21336.204117,68640.564794,178184.300196
observed_future_revenue,4.0,NaN,NaN,NaN,41994.4025,57341.125474,2668.68,6296.8875,19741.52,55439.035,125825.89


Interpretation: consult the corresponding report for mathematical definitions, executed estimates and their limits. Never turn an assumed campaign response into an observed result.

# Retention policy comparison
Risk-only, value-only, risk × value and random targeting are compared at 5%, 10%, 20% contact capacity. Random results average 200 seeded without-replacement samples. Priority is p(inactivity) × estimated future gross revenue, a value-at-risk proxy, not expected treatment benefit. Under homogeneous treatment success and equal costs, selecting largest scores maximizes the scenario objective for a fixed contact count (exchange argument). With heterogeneous costs/uplift, estimate incremental benefit and solve a knapsack/constrained optimization instead. Capacity is a ceiling: a negative net scenario should recommend no contact rather than force spending.

Scenario retained revenue = sum(priority) × assumed_success. Scenario net contribution = retained revenue × assumed_margin − contacts × (contact_cost + incentive). Incentives are charged to every contact. Margin is 30%; success takes 2%, 5%, 10%; costs GBP 1/5/10; incentives GBP 0/10/25. None are observed. An illustrative 10%-capacity, GBP 5 contact, GBP 10 incentive and 5% effectiveness slice:
```
  strategy  capacity  contacts  contact_cost  assumed_success  incentive  assumed_margin  predicted_value_at_risk  scenario_retained_revenue  scenario_net_contribution
    random       0.1        36             5             0.05         10             0.3              5450.551841                 272.527592                -458.241722
      risk       0.1        36             5             0.05         10             0.3              6650.035339                 332.501767                -440.249470
     value       0.1        36             5             0.05         10             0.3              5995.340646                 299.767032                -450.069890
risk_value       0.1        36             5             0.05         10             0.3             10804.388539                 540.219427                -377.934172
```
Risk × value can win its own arithmetic objective by construction; this is not evidence of actual treatment superiority. Test heterogeneous response with a randomized pilot, intention-to-treat analysis, holdout controls, confidence intervals, incremental margin and customer experience guardrails. Churn risk is not persuadability. Conditional lost value may differ from unconditional BG/NBD forecasts; evaluate risk/value overlap before production.
